# Validate bibliography and gazetteer
* Install & set up dependencies
* extract schematron from RNG and transform to XSLT
* run schematron-xslt on files

In [ ]:
import os
import requests
import pandas as pd
from pathlib import Path
import saxonche
from zipfile import ZipFile
from lxml import etree
from datetime import datetime
from zoneinfo import ZoneInfo


In [ ]:
SCRIPT_DIR = Path(__file__).resolve().parent
TMP_DIR = SCRIPT_DIR / "tmp"
LIB_DIR = SCRIPT_DIR / "lib"
VALIDATION_REPORTS_DIR = TMP_DIR / "validationReports"
os.makedirs(TMP_DIR, exist_ok=True)
os.makedirs(LIB_DIR, exist_ok=True)
os.makedirs(VALIDATION_REPORTS_DIR, exist_ok=True)

NAMESPACES = {"tei": "http://www.tei-c.org/ns/1.0"}
XML_NAMESPACE = "http://www.w3.org/XML/1998/namespace"

DATA_HOME = (SCRIPT_DIR / "../..").resolve()

VALIDATION_TARGETS = [
    {
        "section": "geolist",
        "label": "Geolist",
        "path": DATA_HOME / "vicav_geo" / "vicav_geodata.xml",
        "schema": DATA_HOME / "803_RNG_Schematron" / "vicav_geodata.rng",
        "entry_el": "tei:place",
    },
    {
        "section": "bibliography",
        "label": "Bibliography",
        "path": DATA_HOME / "vicav_biblio" / "vicav_biblio_tei_zotero.xml",
        "schema": DATA_HOME / "803_RNG_Schematron" / "vicav_bibliography.rng",
        "entry_el": "tei:biblStruct",
    },
]

ERROR_REPORT = VALIDATION_REPORTS_DIR / "validation_report.html"

with saxonche.PySaxonProcessor(license=False) as proc:
    print(proc.version)
    proc.set_cwd(os.path.dirname(os.path.dirname(os.path.abspath(""))))
    print(proc.cwd)

In [ ]:
def download_and_store(url, force=False):
    fn = os.path.basename(url)
    dl_file_path = str(TMP_DIR / fn)
    if force or not os.path.exists(dl_file_path):
        payload = requests.get(url).content
        with open(dl_file_path, "wb") as handle:
            handle.write(payload)
    return dl_file_path

In [ ]:
def download_and_unzip(url):
    fn = os.path.basename(url)
    ext = Path(fn).suffix
    if ext != ".zip":
        return "not a zip archive"

    zip_file_path = download_and_store(url)
    target_path = str(LIB_DIR / Path(fn).stem)
    payload = requests.get(url).content
    with open(zip_file_path, "wb") as handle:
        handle.write(payload)
    ZipFile(zip_file_path).extractall(path=target_path)
    return target_path

### Install XSL based schematron validator

In [ ]:
_sch_compiler = None

def setup_sch_xslt():
    global _sch_compiler
    if _sch_compiler is not None:
        return _sch_compiler

    sch_dl_url = "https://codeberg.org/SchXslt/schxslt/releases/download/v1.10.1/schxslt-1.10.1-xslt-only.zip"
    sch_home = download_and_unzip(sch_dl_url)
    _sch_compiler = str(Path(sch_home) / "schxslt-1.10.1" / "2.0" / "pipeline-for-svrl.xsl")
    if os.path.exists(_sch_compiler):
        return _sch_compiler
    raise FileNotFoundError(f"Could not locate schematron compiler at {_sch_compiler}")


In [ ]:
setup_sch_xslt()

In [ ]:
def transform(source, stylesheet, output, parameters=None):
    try:
        with saxonche.PySaxonProcessor(license=False) as proc:
            proc.set_configuration_property("xi", "on")
            saxon = proc.new_xslt30_processor()
            params = parameters or {}
            for name, value in params.items():
                saxon.set_parameter(name=name, value=proc.make_string_value(value))
            exec_ = saxon.compile_stylesheet(stylesheet_file=os.path.abspath(stylesheet))
            exec_.apply_templates_returning_file(
                source_file=os.path.abspath(source),
                output_file=os.path.abspath(output),
            )
            if exec_.exception_occurred:
                print(saxon.get_error_message())
                return None
            return output
    except Exception as exc:
        print(f"Python Exception during transform: {exc}")
        return None


## Prepare rng2sch stylesheet

Returns path to the xsl that extracts schematron form the RelaxNG schema.
This should only run once as the file gets locked (by saxon) and so further attempts to pring it to the correct location will fail.

In [ ]:
_rng2sch = None


def setup_rng2sch():
    global _rng2sch
    if _rng2sch is not None:
        return _rng2sch

    rng2sch_dl = "https://raw.githubusercontent.com/Schematron/schematron/master/trunk/converters/code/ToSchematron/ExtractSchFromRNG.xsl"
    dltmp = download_and_store(rng2sch_dl)
    with open(dltmp, encoding="utf-8") as handle:
        lines = handle.read()
    lines = lines.replace("http://www.ascc.net/xml/schematron", "http://purl.oclc.org/dsdl/schematron/")
    lines = lines.replace("<sch:schema", '<sch:schema queryBinding="xslt2"')

    with open(dltmp, "w", encoding="utf-8") as handle:
        handle.write(lines)

    _rng2sch = str(LIB_DIR / os.path.basename(dltmp))
    os.replace(dltmp, _rng2sch)
    return _rng2sch


In [ ]:
setup_rng2sch()

## Extract schematron from RNG and transform to XSLT

In [ ]:
def extractSchematron(rng):
    """extracts a schematron document embedded in an rng schema"""
    print("extracting Schematron document from " + rng)
    rng2sch = setup_rng2sch()
    sch = str(TMP_DIR / f"{Path(rng).name}.sch")
    if not os.path.exists(sch):
        transform(rng, rng2sch, sch)
    return sch

In [ ]:
def compileSchematron(sch):
    output_path = str(TMP_DIR / f"{Path(sch).name}.xsl")
    sch_compiler = setup_sch_xslt()
    transform(sch, sch_compiler, output_path)
    if os.path.exists(output_path):
        return output_path
    raise FileNotFoundError(f"Could not locate compiled stylesheet at {output_path}")

## Run schematron and relaxNG on files

In [ ]:
def schValidate(sch, path, entry_el):
    """Validates a document (at path) against a Schematron schema (at sch). Returns a list of dicts for errors and successful checks."""
    results = []
    out = str(VALIDATION_REPORTS_DIR / os.path.basename(path))
    xsl = compileSchematron(sch)
    try:
        transform(path, xsl, out)
    except saxonche.PySaxonApiError as e:
        print(f"An error occurred while running schValidate on {path}")
        return []

    report = etree.parse(out)
    source_doc = etree.parse(path)
    failedAssert = report.findall("{http://purl.oclc.org/dsdl/svrl}failed-assert")
    successfulReport = report.findall(
        "{http://purl.oclc.org/dsdl/svrl}successful-report"
    )
    for s in successfulReport + failedAssert:
        xpath = (
            s.attrib["location"]
            .replace("Q{http://www.tei-c.org/ns/1.0}", "tei:")
            .replace("Q{}", "")
        )
        msg = s.find("{http://purl.oclc.org/dsdl/svrl}text").text

        resolved_line = None
        entry_id = None
        if xpath:
            try:
                matches = source_doc.xpath(xpath, namespaces=NAMESPACES)
                if matches:
                    first = matches[0]
                    if isinstance(first, etree._Element):
                        resolved_line = first.sourceline
                    elif hasattr(first, "getparent") and first.getparent() is not None:
                        resolved_line = first.getparent().sourceline
                           # Find nearest ancestor-or-self entry element with xml:id
                    entry_xpath = f"{xpath}/ancestor-or-self::{entry_el}[1]"
                    entry_matches = source_doc.xpath(entry_xpath, namespaces=NAMESPACES)
                    if entry_matches:
                        entry_id = entry_matches[0].get(f"{{{XML_NAMESPACE}}}id")
            except Exception:
                resolved_line = None
        
        results.append(
            {
                "type": "error",
                "message": msg,
                "line": resolved_line,
                "location": xpath,
                "source": path,
                "entry_id": entry_id,
                "stage": "schematron",
                "exceptionType": str(s.tag).replace(
                    "{http://purl.oclc.org/dsdl/svrl}", ""
                ),
            }
        )
    return results

In [ ]:
def validate_document(target):
    results = []
    doc_path = Path(target["path"])
    schema_path = Path(target["schema"])
    entry_el = target.get("entry_el", "tei:place")

    if not doc_path.exists():
        return [{
            "type": "error",
            "message": f"Document not found: {doc_path}",
            "line": None,
            "location": "n/a",
            "entry_id": None,
            "stage": "file",
            "exceptionType": "FileNotFoundError",
        }]

    doc = None
    try:
        doc = etree.parse(str(doc_path))
    except etree.XMLSyntaxError as exc:
        results.append({
            "type": "error",
            "message": str(exc),
            "line": exc.lineno,
            "location": "n/a",
            "entry_id": None,
            "stage": "parsing",
            "exceptionType": type(exc).__name__,
        })

    if doc is not None:
        try:
            relaxng_doc = etree.parse(str(schema_path))
            print(f"Parsed RNG schema {schema_path} successfully.")
            relaxng = etree.RelaxNG(relaxng_doc)
            relaxng.assertValid(doc)

        except etree.DocumentInvalid as exc:
            for error in exc.error_log:
                if error.message != "Invalid attribute schemaLocation for element TEI":
                    entry_xpath = f"{error.path}/ancestor-or-self::{entry_el}[1]"
                    if entry_xpath is not None and doc.xpath(entry_xpath, namespaces=NAMESPACES):
                        entry = doc.xpath(entry_xpath, namespaces=NAMESPACES)[0]
                        entry_id = entry.get("{http://www.w3.org/XML/1998/namespace}id")
                    else:
                        entry_id = ""
                    results.append({
                        "type": "error",
                        "message": error.message,
                        "line": error.line,
                        "location": "n/a" if error.path is None else error.path,
                        "entry_id": entry_id,
                        "stage": "relaxng",
                        "exceptionType": type(exc).__name__,
                    })
        except etree.RelaxNGValidateError as exc:
            for error in exc.error_log:
                results.append({
                    "type": "error",
                    "message": str(error),
                    "line": getattr(error, "line", None),
                    "location": getattr(error, "path", "n/a"),
                    "entry_id": None,
                    "stage": "relaxng",
                    "exceptionType": type(exc).__name__,
                })
        except etree.RelaxNGError as e:
            print(f"RelaxNG validation failed: {e}")
        schematron_path = extractSchematron(str(schema_path))
        schematron_xsl = compileSchematron(schematron_path)
        transform_result = transform(str(doc_path), schematron_xsl, str(VALIDATION_REPORTS_DIR / f"{doc_path.stem}.svrl"))

        if transform_result is not None:
            try:
                schematron_errors = schValidate(schematron_path, str(doc_path), entry_el)
            except Exception as exc:
                results.append({
                    "type": "error",
                    "message": f"Schematron execution failed: {exc}",
                    "line": None,
                    "location": "n/a",
                    "entry_id": None,
                    "stage": "schematron",
                    "exceptionType": type(exc).__name__,
                })
                return results

            results.extend(schematron_errors)

    return results

In [ ]:
def make_clickable(source, line=None):
    try:
        relative_path = Path(source).resolve().relative_to(DATA_HOME)
        link = f"https://github.com/acdh-oeaw/vicav-library/blob/main/{relative_path.as_posix()}"
    except ValueError:
        link = f"https://github.com/acdh-oeaw/vicav-library/blob/main/{Path(source).name}"

    if line:
        return f'<a href="{link}#L{line}" target="_blank">{source}</a>'
    return f'<a href="{link}" target="_blank">{source}</a>'


In [ ]:
validation_html_begin = """<!doctype html>
<html>
<head>
    <meta charset="utf-8" />
    <title>WIBARAB validation report</title>
    <style>
    body { font-family: Arial, sans-serif; font-size: 13px; margin: 20px; }
    table { font-family: Arial, sans-serif; font-size: 13px; border-collapse: collapse; width: 100%; }
    th, td { border: 1px solid #ccc; padding: 6px; text-align: left; min-width: 50px; max-width: 300px; overflow-wrap: break-word; word-break: break-word; }
    thead { background: #eee; }
    #tableFilter { margin-bottom: 12px; }
    details.file-group { margin-bottom: 8px; border: 1px solid #ccc; border-radius: 4px; }
    details.file-group summary { background: #eee; padding: 8px 12px; cursor: pointer; font-weight: bold; list-style: none; user-select: none; }
    details.file-group summary::-webkit-details-marker { display: none; }
    details.file-group summary::before { content: "▶  "; font-size: 10px; }
    details.file-group[open] summary::before { content: "▼  "; }
    .file-errors { padding: 8px; }
    .error-count { font-weight: normal; color: #c00; margin-left: 8px; }
    .file-path { font-weight: normal; color: #666; margin-left: 8px; font-size: 11px; }
    #resultSummary { margin-left: 12px; color: #555; font-size: 12px; }
    </style>
</head>
<body>
<h1>WIBARAB validation report</h1>
"""

validation_filter_ui = """<label for="tableFilter">Filter rows:</label>
<input id="tableFilter" type="text" placeholder="Type to filter rows or ids" onkeyup="filterValidationTable()" style="margin-left:8px;width:320px" />
<button onclick="document.querySelectorAll('details.file-group').forEach(d=>d.open=true)" style="margin-left:8px">Expand all</button>
<button onclick="document.querySelectorAll('details.file-group').forEach(d=>d.open=false)" style="margin-left:4px">Collapse all</button>
<span id="resultSummary"></span>
<div style="margin-bottom:12px"></div>
"""

validation_filter_script = """<script>
function updateSummary(visibleRows, totalRows, visibleFiles, totalFiles) {
    const el = document.getElementById("resultSummary");
    if (!el) return;
    if (visibleRows === totalRows) {
        el.textContent = totalRows + " error" + (totalRows !== 1 ? "s" : "") + " in " + totalFiles + " file" + (totalFiles !== 1 ? "s" : "");
    } else {
        el.textContent = visibleRows + " of " + totalRows + " error" + (totalRows !== 1 ? "s" : "") + " in " + visibleFiles + " of " + totalFiles + " section" + (totalFiles !== 1 ? "s" : "");
    }
}

function filterValidationTable() {
    const input = document.getElementById("tableFilter");
    const filter = input.value.toLowerCase();
    const groups = document.querySelectorAll("details.file-group");
    let totalRows = 0, visibleRows = 0, totalFiles = groups.length, visibleFiles = 0;

    groups.forEach(function(group) {
        const summaryTxt = group.querySelector("summary").textContent.toLowerCase();
        const rows = group.querySelectorAll("tbody tr");
        let anyRowVisible = false;
        totalRows += rows.length;

        rows.forEach(function(row) {
            const txt = row.textContent || row.innerText;
            const visible = !filter || txt.toLowerCase().indexOf(filter) > -1;
            row.style.display = visible ? "" : "none";
            if (visible) { anyRowVisible = true; visibleRows++; }
        });

        if (filter && summaryTxt.indexOf(filter) > -1) {
            rows.forEach(function(row) {
                if (row.style.display === "none") { row.style.display = ""; visibleRows++; }
            });
            anyRowVisible = true;
        }

        group.style.display = (!filter || anyRowVisible) ? "" : "none";
        if (anyRowVisible) visibleFiles++;
        if (filter && anyRowVisible) group.open = true;
    });

    updateSummary(visibleRows, totalRows, visibleFiles, totalFiles);
}

window.addEventListener("DOMContentLoaded", function() {
    filterValidationTable();
});
</script>
"""

validation_html_end = """</body></html>"""


In [ ]:
def write_report(out_file, records):
    df_err = pd.DataFrame(records)
    if df_err.empty:
        with open(out_file, "w", encoding="utf-8") as handle:
            handle.write(validation_html_begin)
            handle.write("<p>No validation errors found.</p>")
            handle.write(validation_html_end)
        return

    df_err["link"] = df_err.apply(lambda row: make_clickable(row["source"], row.get("line")), axis=1)
    for col in ("entry", "relativePath", "type"):
        if col in df_err.columns:
            df_err.drop(columns=[col], inplace=True)

    display_cols = [col for col in ("entry_id", "message", "stage", "location", "line", "link") if col in df_err.columns]
    html_parts = []
    for section, group in df_err.groupby("section", sort=False):
        label = group["label"].iloc[0]
        count = len(group)
        html_parts.append('<details class="file-group">')
        html_parts.append(
            f'<summary>{label}'
            f'<span class="error-count">({count} error{"s" if count != 1 else ""})</span>'
            f'<span class="file-path">{group["source"].iloc[0]}</span></summary>'
        )
        html_parts.append('<div class="file-errors">')
        html_parts.append(group[display_cols].to_html(render_links=True, escape=False, index=False))
        html_parts.append("</div></details>")

    with open(out_file, "w", encoding="utf-8") as handle:
        handle.write(validation_html_begin)
        handle.write(f'<p style="color:#888;font-size:12px;">Generated: {datetime.now(ZoneInfo("Europe/Vienna")).strftime("%Y-%m-%d %H:%M:%S %Z")}</p>\n')
        handle.write(validation_filter_ui)
        handle.write("\n".join(html_parts))
        handle.write(validation_filter_script)
        handle.write(validation_html_end)


In [ ]:
all_errors = []
for target in VALIDATION_TARGETS:
    print(f"Validating {target['label']} -> {target['path']}")
    validation_results = validate_document(target)
    for result in validation_results:
        result.setdefault("section", target["section"])
        result.setdefault("label", target["label"])
        result.setdefault("source", str(target["path"]))
        all_errors.append(result)

write_report(ERROR_REPORT, all_errors)
print(f"Wrote report to {ERROR_REPORT}")